# Coastal flood step 02: direct damages

This notebook runs direct damage calculations from intersection outputs generated in step 01.

- Input intersections: `results/*_splits__coastal_flood_rasters_fixed_for_intersections__*.geoparquet`
- Output damages: `results/direct_damages/*/*_direct_damages_parameter_set_0.parquet`


In [ ]:
from pathlib import Path
import subprocess
import pandas as pd


In [ ]:
# Core paths
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
output_path = base_path / 'dphil_paper_3/results'
shared_intersections_path = base_path / 'dphil_paper_3/results/coastal_flood_network_intersections'
shared_intersections_path.mkdir(parents=True, exist_ok=True)
data_root = base_path / 'dphil_common_cross_cutting/common_incoming_data'
damage_curves_path = data_root / 'damage_curves'

network_csv = data_root / 'networks/network_layers_hazard_intersections_details.csv'
damage_curves_csv = damage_curves_path / 'asset_damage_curve_mapping.csv'
hazard_damage_parameters_csv = damage_curves_path / 'hazard_damage_parameters.csv'
script_path = base_path / 'scripts/analysis/damage_calculations.py'

# Intersections/hazard list from step 01
hazard_csv = shared_intersections_path / 'coastal_flood_rasters_for_intersections.csv'
raster_details_csv = hazard_csv
hazard_layers_name = raster_details_csv.stem

for p in [network_csv, hazard_csv, damage_curves_csv, hazard_damage_parameters_csv, script_path]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

print("Using:")
print("- network_csv:", network_csv)
print("- hazard_csv:", hazard_csv)
print("- script_path:", script_path)


In [ ]:
def resolve_network_asset_file(asset_relative_path: str) -> Path:
    rel = Path(asset_relative_path)
    p1 = data_root / rel
    p2 = data_root / 'networks' / rel
    if p1.exists():
        return p1
    if p2.exists():
        return p2
    raise FileNotFoundError(f'Could not resolve asset path: {asset_relative_path}\nChecked: {p1} and {p2}')

damage_results_folder = output_path / 'direct_damages'
damage_results_folder.mkdir(parents=True, exist_ok=True)

# Sensitivity set for this baseline direct-damage run
sensitivity_csv = output_path / 'sensitivity_parameters.csv'
pd.DataFrame([
    {'cost_uncertainty_parameter': 0.0, 'damage_uncertainty_parameter': 0.0}
]).to_csv(sensitivity_csv, index=False)
print('Saved sensitivity file:', sensitivity_csv)


In [ ]:
# Run direct damage calculations per asset layer
asset_data_details = pd.read_csv(network_csv)
failures = []
runs = 0

for asset_info in asset_data_details.itertuples(index=False):
    asset_gpkg_file = resolve_network_asset_file(asset_info.path)
    intersection_file = shared_intersections_path / f'{asset_info.asset_gpkg}_splits__{hazard_layers_name}__{asset_info.asset_layer}.geoparquet'
    output_file = damage_results_folder / f'{asset_info.asset_gpkg}_{asset_info.asset_layer}' / f'{asset_info.asset_gpkg}_{asset_info.asset_layer}_direct_damages_parameter_set_0.parquet'
    output_file.parent.mkdir(parents=True, exist_ok=True)

    if not intersection_file.exists():
        failures.append((asset_info.asset_gpkg, asset_info.asset_layer, f'Missing intersection: {intersection_file}'))
        continue

    args = [
        'python', str(script_path),
        '--network-csv', str(network_csv),
        '--hazard-csv', str(hazard_csv),
        '--sensitivity-csv', str(sensitivity_csv),
        '--sensitivity-id', '0',
        '--asset-gpkg-file', str(asset_gpkg_file),
        '--asset-gpkg-label', str(asset_info.asset_gpkg),
        '--asset-layer', str(asset_info.asset_layer),
        '--damage-curve-mapping-csv', str(damage_curves_csv),
        '--damage-threshold-uplift-csv', str(hazard_damage_parameters_csv),
        '--damage-curves-dir', str(damage_curves_path),
        '--intersection', str(intersection_file),
        '--output-path', str(output_file),
    ]

    print(f'Running: {asset_info.asset_gpkg} | {asset_info.asset_layer}')
    result = subprocess.run(args, capture_output=True, text=True)
    if result.returncode != 0:
        failures.append((asset_info.asset_gpkg, asset_info.asset_layer, result.stderr.strip() or 'Unknown error'))
        print(result.stdout)
        print(result.stderr)
    else:
        runs += 1

print(f'Completed successful runs: {runs}')
print(f'Failures: {len(failures)}')
if failures:
    pd.DataFrame(failures, columns=['asset_gpkg', 'asset_layer', 'error']).head(20)


In [ ]:
# Quick output check
damage_files = sorted((output_path / 'direct_damages').glob('*/*_direct_damages_parameter_set_0.parquet'))
print('Direct-damage parquet files found:', len(damage_files))
for p in damage_files[:15]:
    print('-', p)
